# 08 Fine-tuning (stage 5): teach a model to read reviews, not tweets
Goal: beat the off-the-shelf baseline (three-way macro F1 0.571) on the same held-out test set.

## Step 1. A fair test set
- Remove duplicate texts so the same sentence cannot appear in both training and test.
- Split 80/20, keeping the same mix of negative, neutral and positive in both parts (stratified).
- Recompute the baseline on exactly this test set.

In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE = r".."
PROC = BASE + r"\data\processed"

data = pd.read_parquet(PROC + r"\reviews_sentiment.parquet")
data['text_norm'] = data['review_comment_message'].str.lower().str.strip()
print("Comments:", len(data), "| unique texts:", data['text_norm'].nunique())
data = data.drop_duplicates('text_norm').reset_index(drop=True)

train, test = train_test_split(data, test_size=0.2, stratify=data['star_label'], random_state=42)
print("Train:", len(train), "| Test:", len(test))
print(test['star_label'].value_counts(normalize=True).round(3), "\n")

print("BASELINE on the test set")
print(classification_report(test['star_label'], test['sentiment'], digits=3))

train.to_parquet(PROC + r"\s5_train.parquet")
test.to_parquet(PROC + r"\s5_test.parquet")

Comments: 40775 | unique texts: 34912
Train: 27929 | Test: 6983
star_label
positive    0.615
negative    0.294
neutral     0.091
Name: proportion, dtype: float64 

BASELINE on the test set
              precision    recall  f1-score   support

    negative      0.687     0.638     0.662      2050
     neutral      0.158     0.411     0.228       638
    positive      0.957     0.761     0.848      4295

    accuracy                          0.693      6983
   macro avg      0.601     0.603     0.579      6983
weighted avg      0.805     0.693     0.737      6983



**Fair test set.** 5,863 duplicate texts removed (14.4%). Train 27,929, test 6,983 (positive 61.5%, negative 29.4%, neutral 9.1%).
**Baseline on this test set:** macro F1 0.579, accuracy 69.3%.
- Negative: recall 0.638, so 36% of complaints are missed (mostly factual complaints read as neutral).
- Neutral: F1 0.228; the model predicts neutral far too often (precision 0.158).
- Positive: F1 0.848.
Targets: higher macro F1, and above all higher negative recall (complaints caught). Neutral will stay hard: 3-star reviews are genuinely mixed.

## Step 2. Quick model: embeddings + logistic regression
Sentence embeddings (paraphrase-multilingual-MiniLM-L12-v2) as features, logistic regression trained on the star-based labels. The embedding model is not changed (no fine-tuning yet); only the classifier on top learns from reviews.
class_weight='balanced' gives the small neutral class equal weight.

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

start = time.time()
X_tr = embedder.encode(train['review_comment_message'].tolist(), batch_size=64,
                       show_progress_bar=True, normalize_embeddings=True)
X_te = embedder.encode(test['review_comment_message'].tolist(), batch_size=64,
                       show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding: {time.time() - start:.0f} s")
np.save(PROC + r"\s5_X_train.npy", X_tr)
np.save(PROC + r"\s5_X_test.npy", X_te)

clf_lr = LogisticRegression(max_iter=2000, class_weight='balanced')
clf_lr.fit(X_tr, train['star_label'])
pred_lr = clf_lr.predict(X_te)

print("\nQUICK MODEL on the test set")
print(classification_report(test['star_label'], pred_lr, digits=3))

Embedding: 293 s

QUICK MODEL on the test set
              precision    recall  f1-score   support

    negative      0.779     0.729     0.753      2050
     neutral      0.226     0.503     0.312       638
    positive      0.956     0.811     0.878      4295

    accuracy                          0.759      6983
   macro avg      0.654     0.681     0.647      6983
weighted avg      0.837     0.759     0.789      6983



**Result: the quick model beats the baseline on the same test set.**
| | Baseline | Embeddings + logistic regression |
| --- | --- | --- |
| Macro F1 | 0.579 | 0.647 |
| Accuracy | 69.3% | 75.9% |
| Negative recall | 0.638 | 0.729 |
| Negative precision | 0.687 | 0.779 |
| Neutral F1 | 0.228 | 0.312 |
Learning from real reviews catches about 186 more of the 2,050 complaints in the test set and flags complaints more accurately. Neutral stays hard. New bar for full fine-tuning: macro F1 0.647.

In [3]:
%pip install gradio